### FORMULACIÓN DEL PROBLEMA

In [4]:
# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial 
        self.goal = goal 
    
    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state
    
    def h(self, state):
        return 0

In [5]:
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph
    
    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    def result(self, state, action):
        return action

In [6]:
class Node:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        return Node(next_state, self)

### ALGORITMO HILL CLIMBING

In [7]:
class HeuristicGraphProblem(GraphProblem):
    def __init__(self, initial, goal, graph, heuristics):
        super().__init__(initial, goal, graph)
        self.heuristics = heuristics
# Devuelev el valor heurístico del estdao. 
# Si el estado no existe en el diccionario, devuelve infinito para evitarlo.        
    def h(self, state):
        return self.heuristics.get(state, float('inf'))

In [8]:
def hill_climbing(problem):
    # Comenzamos evaluando el nodo inicial
    current = Node(problem.initial)
    visited = {current.state}
    while True:
        # Si ya llegamos a la meta, terminamos
        if problem.is_goal(current.state):
            return current
            
        # Obtenemos los vecinos
        neighbors = current.expand(problem)
        # Eliminamos los nodos que ya hemos visitado
        neighbors = [
            node for node in neighbors
            if node.state not in visited
        ]
        
        # Si no hay salida que sea un callejón sin salida, nos detenemos
        if not neighbors:
            return current
            
        # Encontramos al vecino con la menor heurística el que parece estar más cerca de la meta
        best_neighbor = min(neighbors, key=lambda node: problem.h(node.state))
        
        # Si el mejor vecino no mejora nuestra situación actual osea su heurística es mayor o igual, nos atascamos en un óptimo local y terminamos.
        if problem.h(best_neighbor.state) >= problem.h(current.state):
            return current
            
        # Si el vecino es mejor, nos movemos hacia él y repetimos
        current = best_neighbor
        visited.add(current.state)

### PRUEBA

In [9]:
if __name__ == "__main__":
    # 3 pasos en línea recta
    mapa_facil = {
        'Inicio': {'Camino Falso': 1, 'Puerta': 1},
        'Camino Falso': {'Inicio': 1},
        'Puerta': {'Inicio': 1, 'Meta': 1},
        'Meta': {'Puerta': 1}
    }

    heuristica_facil = {
        'Inicio': 10,
        'Camino Falso': 20,  # Numero alto = Mal camino
        'Puerta': 5,         # Numero bajo = Buen camino
        'Meta': 0            # La Meta siempre va valer 0
    }

    problema = HeuristicGraphProblem('Inicio', 'Meta', mapa_facil, heuristica_facil)
    resultado = hill_climbing(problema)

    if resultado:
        camino = " -> ".join(resultado.path())
        print(f"Ruta tomada: {camino}")
        
        if problema.is_goal(resultado.state):
            print("Llego a la meta con exito")
        else:
            print("Se atascó en un óptimo local.")

Ruta tomada: Inicio -> Puerta -> Meta
Llego a la meta con exito
